# 06 — Training di Colab

**Tujuan:** menjalankan smoke test lalu full training IndoBERT dari Colab,
menyimpan checkpoint ke Drive, dan mengevaluasi validation set. Semua logika
tetap di `src/cyberbully/`; notebook ini hanya memanggil CLI.

**Prasyarat:** `00_colab_setup.ipynb` sudah selesai (data/processed ada), runtime
GPU aktif, dan Drive ter-mount.

**Estimasi waktu:** smoke test 5–10 menit; full training 30–60 menit di T4.
Colab gratis bisa memutus koneksi saat idle — checkpoint ke Drive adalah
pengamannya.

**Aturan:** test set **tidak** dievaluasi di notebook ini. Threshold dan
hyperparameter dipilih dari validation; test hanya sekali untuk laporan akhir.

## Glosarium

- **Epoch**: satu putaran penuh melewati seluruh data training.
- **Batch size**: jumlah contoh per langkah update; besar = cepat tapi butuh
  memori lebih banyak.
- **Learning rate**: besar langkah update; terlalu besar membuat loss meledak,
  terlalu kecil membuat training lambat.
- **Overfitting**: model hafal train tetapi jelek di data baru. Tanda: loss train
  turun, F1 validation stagnan/turun.
- **F1**: rata-rata harmonik precision dan recall untuk kelas bully.
- **Checkpoint**: salinan model di akhir epoch agar training bisa dilanjutkan.

## 1. Cek runtime dan tentukan path

Sel ini mendeteksi Colab, memilih GPU bila ada, dan menentukan folder artefak.
Di Colab, model dan metrik disimpan ke Drive; di Mac, ke folder repo yang sudah
gitignored. Mengimpor `ROOT` dari `config` membuat path tetap benar walau
notebook dijalankan dari folder lain.

In [ ]:
import os
import sys
from pathlib import Path

import torch

IN_COLAB = "google.colab" in sys.modules
try:
    from cyberbully.config import OUT_DIR, ROOT
except ModuleNotFoundError:
    ROOT = Path("/content/CyberBully")
    OUT_DIR = ROOT / "out"

DRIVE_DIR = Path("/content/drive/MyDrive/cyberbully")
if IN_COLAB:
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    ARTIFACT_ROOT = DRIVE_DIR
else:
    ARTIFACT_ROOT = OUT_DIR

MODEL_DIR = ARTIFACT_ROOT / "models" / "indobert"
RUN_DIR = ARTIFACT_ROOT / "runs"

print("Di Colab :", IN_COLAB)
print("Repo     :", ROOT)
print("Artefak  :", ARTIFACT_ROOT)
print("Device   :", "cuda" if torch.cuda.is_available() else "mps/cpu")

## 2. Smoke test

Smoke test memakai model **kecil dan acak**. Tujuannya hanya membuktikan data,
tokenizer, Trainer, dan penyimpanan model bekerja. F1-nya tidak bermakna dan
tidak boleh dilaporkan sebagai performa. Jika langkah ini gagal, jangan lanjut
ke training IndoBERT — perbaiki dulu error-nya.

In [ ]:
import subprocess

RUN_SMOKE = True
SMOKE_DIR = ROOT / "models" / "smoke"

if RUN_SMOKE:
    smoke_cmd = [
        sys.executable, "-m", "cyberbully.train",
        "--model", "hf-internal-testing/tiny-random-BertForSequenceClassification",
        "--subset", "256", "--epochs", "1",
        "--batch-size", "16", "--max-length", "64",
        "--out", str(SMOKE_DIR),
    ]
    result = subprocess.run(smoke_cmd, cwd=ROOT, text=True)
    print("exit code:", result.returncode)
else:
    print("Smoke test dilewati.")

## 3. Full training IndoBERT

Sekarang model asli: `indobenchmark/indobert-base-p1`, 3 epoch, `max_length=128`.
`--class-weights` memberi bobot lebih besar pada kelas minoritas. `--out`
diarahkan ke Drive agar checkpoint selamat saat runtime mati. Set
`RUN_FULL = True` hanya setelah smoke test lolos.

Perkiraan T4: 30–60 menit. Jika GPU kehabisan memori, turunkan batch size dan
pakai gradient accumulation (`--batch-size 8 --grad-accum 2 --max-length 96`).

In [ ]:
RUN_FULL = False

if RUN_FULL:
    train_cmd = [
        sys.executable, "-m", "cyberbully.train",
        "--model", "indobenchmark/indobert-base-p1",
        "--epochs", "3", "--batch-size", "16", "--max-length", "128",
        "--class-weights",
        "--out", str(MODEL_DIR),
    ]
    result = subprocess.run(train_cmd, cwd=ROOT, text=True)
    print("exit code:", result.returncode)
else:
    print("Full training dilewati. Ubah RUN_FULL = True setelah smoke test lolos.")

## 4. Evaluasi validation

Evaluasi memakai validation set, bukan test. Threshold awal 0.5; nanti
`05_threshold_tuning` memilih threshold dari kurva precision-recall validation.
Perhatikan precision/recall kelas bully, bukan hanya accuracy.

In [ ]:
if (MODEL_DIR / "config.json").exists():
    eval_cmd = [
        sys.executable, "-m", "cyberbully.evaluate",
        "--model-dir", str(MODEL_DIR),
        "--split", "val",
        "--out-dir", str(RUN_DIR / "val"),
    ]
    result = subprocess.run(eval_cmd, cwd=ROOT, text=True)
    print("exit code:", result.returncode)
else:
    print("Model belum ada; jalankan full training dulu.")

## 5. Cek artefak

Setelah training, folder model harus memuat `run_meta.json`, `val_metrics.json`,
`val_predictions.csv`, dan checkpoint. Salinan di Drive adalah bukti utama;
jangan mengandalkan `/content` yang hilang saat runtime berhenti. Catat waktu
per epoch dan versi `transformers` dari `run_meta.json` ke laporan.

In [ ]:
if MODEL_DIR.exists():
    for path in sorted(MODEL_DIR.glob("*")):
        print(path.name, "(folder)" if path.is_dir() else f"{path.stat().st_size:,} byte")
else:
    print("Belum ada artefak model.")

## Ringkasan

- Smoke test membuktikan pipeline berjalan; metriknya tidak dilaporkan.
- Full training memakai validation untuk memantau F1, bukan test.
- Checkpoint dan metrik berada di Drive; `run_meta.json` mencatat versi dan
  konfigurasi untuk reproduksibilitas.
- Langkah berikutnya: `05_threshold_tuning` dari `val_predictions.csv`, lalu
  evaluasi test sekali dengan `--confirm-test`.